## 8 Ball Table Analyses - Task 1 Computer Vision

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
import colorsys


In [ ]:
IMG_DIR = Path("development_set/")
OUTPUT_DIR= Path("output/top_views")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg",".jpeg", ".png", ".bmp"}
image_paths = sorted([ path for path in IMG_DIR.iterdir()
                       if path.suffix.lower() in VALID_EXTENSIONS])

print(f"Found {len(image_paths)} images")
for i, path in enumerate(image_paths[:5]):
    print(f"[{i}] {path.name}")


In [ ]:
def show_images_grid(images, titles=None, figsize_per_row=(18, 5)):
    if isinstance(images, np.ndarray):
        images = [images]

    if isinstance(titles, str):
        titles = [titles]

    num_images = len(images)

    if num_images == 0:
        print("No images provided to display.")
        return

    cols = min(3, num_images)

    rows = math.ceil(num_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_row[0], figsize_per_row[1] * rows))

    if hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < num_images:
            img = images[i]

            if len(img.shape) == 3:
                img_to_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_to_show)
            else:
                ax.imshow(img, cmap="gray")

            if titles and i < len(titles):
                ax.set_title(titles[i])
            else:
                ax.set_title(f"Image {i}")

            ax.axis("off")

        else:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
if not image_paths:
    raise FileNotFoundError(f"No image files found in {IMG_DIR}")

paths_to_load = image_paths[:18]

loaded_images = []
image_titles = []

for path in paths_to_load:
    img = cv2.imread(str(path))

    if img is None:
        print(f"Could not load image: {path}")
        continue

    loaded_images.append(img)
    image_titles.append(f"Original: {path.name}")

    print(f"Loaded: {path.name} | Shape: {img.shape}")

if not loaded_images:
    raise FileNotFoundError("Failed to load any of the selected images.")

show_images_grid(loaded_images, titles=image_titles)

# Top View of table

In [ ]:
TARGET_HSV = (102, 140, 0)

In [ ]:
def isolate_table_color(image, target_hsv=TARGET_HSV, tol_h=5, tol_s=60):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, _ = target_hsv

    lower = np.array([max(h - tol_h, 0),
                      max(s - tol_s, 0),
                      40],
                     dtype=np.uint8)

    upper = np.array([min(h + tol_h, 179),
                      min(s + tol_s, 255),
                      255],
                     dtype=np.uint8)

    mask = cv2.inRange(hsv, lower, upper)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    return mask

In [ ]:
table_masks = []

for img in loaded_images:
    single_mask = isolate_table_color(img)

    table_masks.append(single_mask)

show_images_grid(table_masks, image_titles)

In [ ]:
def show_images_mask_grid(images, masks, titles=None, figsize_per_row=(18, 5)):
    num_pairs = len(images)
    total_images = num_pairs * 2

    cols = min(3, total_images)
    rows = math.ceil(total_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(20, 4* rows))

    if hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for i in range(num_pairs):
        orig_idx = i * 2
        mask_idx = i * 2 + 1

        img = images[i]
        if len(img.shape) == 3:
            img_to_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[orig_idx].imshow(img_to_show)
        else:
            axes[orig_idx].imshow(img, cmap="gray")

        axes[orig_idx].set_title(f"Original {i}")
        axes[orig_idx].axis("off")

        mask = masks[i]
        axes[mask_idx].imshow(mask, cmap="gray")
        axes[mask_idx].set_title(f"Mask {i}")
        axes[mask_idx].axis("off")

    for j in range(total_images, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

show_images_mask_grid(loaded_images, table_masks, image_titles)

In [ ]:
def get_table_contour(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        print("No contours found in the mask!")
        return None

    table_contour = max(contours, key=cv2.contourArea)
    return table_contour

In [ ]:
contour_images = []
contour_titles = []
my_contours = []

for i, (img, mask) in enumerate(zip(loaded_images, initial_view_masks)):
    contour_img = img.copy()
    table_contour = get_table_contour(mask)

    my_contours.append(table_contour)

    if table_contour is not None:
        cv2.drawContours(contour_img, [table_contour], -1, (0, 255, 0), 3)
        contour_titles.append(f"Table Contour {i+1}")
    else:
        contour_titles.append(f"No Contour {i+1}")

    contour_images.append(contour_img)

show_images_grid(contour_images, titles=contour_titles)

In [ ]:
def order_points(pts):
    pts = np.array(pts, dtype="float32")
    center = np.mean(pts, axis=0)

    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    pts = pts[np.argsort(angles)]

    s = pts.sum(axis=1)
    start = np.argmin(s)
    pts = np.roll(pts, -start, axis=0)

    return pts

In [ ]:
def get_table_corners(table_contour, padding=0):
    hull = cv2.convexHull(table_contour)
    peri = cv2.arcLength(hull, True)

    pts = None
    for eps in np.arange(0.005, 0.10, 0.002):
        approx = cv2.approxPolyDP(hull, eps * peri, True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype("float32")
            break

    if pts is None:
        rect = cv2.minAreaRect(hull)
        pts = cv2.boxPoints(rect).astype("float32")

    ordered = order_points(pts)

    if padding > 0:
        center = np.mean(ordered, axis=0)
        padded = []
        for pt in ordered:
            direction = pt - center
            unit_direction = direction / np.linalg.norm(direction)
            padded.append(pt + unit_direction * padding)
        return np.array(padded, dtype="float32")

    return ordered

In [ ]:
corner_images = []
corner_titles = []
my_corners = []

for i, (img, contour) in enumerate(zip(loaded_images, my_contours)):
    corner_img = img.copy()

    if contour is not None:
        ordered_box = get_table_corners(contour)
        my_corners.append(ordered_box)


        draw_box = np.intp(ordered_box)
        cv2.drawContours(corner_img, [draw_box], -1, (0, 0, 255), 3)

        corner_titles.append(f"Padded Corners {i+1}")
    else:
        my_corners.append(None)
        corner_titles.append(f"No Table {i+1}")

    corner_images.append(corner_img)

show_images_grid(corner_images, titles=corner_titles)

In [ ]:
def get_top_view(image, src_corners, width=1000, height=500):
    (tl, tr, br, bl) = src_corners

    width_top = np.linalg.norm(tr - tl)
    height_left = np.linalg.norm(bl - tl)

    if height_left > width_top:
        src_corners = np.array([bl, tl, tr, br], dtype="float32")

    dst_corners = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(src_corners, dst_corners)
    top_view = cv2.warpPerspective(image, M, (width, height))

    return top_view

In [ ]:
top_view_images = []
top_view_titles = []

for i, (img, corners) in enumerate(zip(loaded_images, my_corners)):

    if corners is not None:
        flat_table = get_top_view(img, corners)

        top_view_images.append(flat_table)
        top_view_titles.append(f"Top View {i+1}")
    else:
        top_view_images.append(np.zeros((600, 800, 3), dtype=np.uint8))
        top_view_titles.append(f"Failed {i+1}")

show_images_grid(top_view_images, titles=top_view_titles)

# Get the balls

In [ ]:
initial_view_masks = []

for img in top_view_images:
    single_mask = isolate_table_color(img)


    initial_view_masks.append(single_mask)

show_images_mask_grid(top_view_images, initial_view_masks, image_titles)

In [ ]:
felt_masks = []

for mask in initial_view_masks:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (40, 40))
    felt_mask = cv2.erode(mask, kernel, iterations=2)
    felt_masks.append(felt_mask)

show_images_mask_grid(top_view_images, felt_masks, image_titles)

In [ ]:
def detect_balls(image, felt_mask, min_area=150, max_area=3000, min_circularity=0.55):
    ball_mask = cv2.bitwise_not(felt_mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    ball_mask = cv2.morphologyEx(ball_mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(ball_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    circles = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area or area > max_area:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        circularity = 4 * math.pi * area / (perimeter ** 2)
        if circularity < min_circularity:
            continue

        (x, y), r = cv2.minEnclosingCircle(cnt)
        circles.append((int(x), int(y), int(r)))

    return circles


In [ ]:
ball_detections = []
detection_images = []

for img, felt_mask in zip(top_view_images, felt_masks):
    circles = detect_balls(img, felt_mask)
    ball_detections.append(circles)

    vis = img.copy()
    for (x, y, r) in circles:
        cv2.circle(vis, (x, y), r, (0, 255, 0), 2)
        cv2.circle(vis, (x, y), 2, (0, 0, 255), 3)
    detection_images.append(vis)

show_images_grid(detection_images, top_view_titles)
